In [107]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv(Path.cwd().parent.parent / "archive" / "transactions_data.csv")

In [108]:
df.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


In [109]:
def clean_transactions(transactions):
    transactions = transactions.copy()

    transactions['date'] = pd.to_datetime(transactions['date'])
    transactions['amount'] = transactions['amount'].str.replace('$', '').astype(float)
    transactions['use_chip'] = transactions['use_chip'].astype('category')
    transactions['merchant_state'] = transactions['merchant_state'].fillna('Online').astype('category')
    transactions['errors'] = transactions['errors'].fillna('None').astype('category')
    transactions.drop(columns=['merchant_id', 'merchant_city', 'merchant_state', 'zip'], inplace=True)

    # One-hot encode 'use_chip' column
    encoder = OneHotEncoder(sparse_output=False)
    use_chip_encoded = encoder.fit_transform(transactions[['use_chip']])
    encoded_cols = encoder.get_feature_names_out(['use_chip'])
    encoded_cols_df = pd.DataFrame(use_chip_encoded, columns=encoded_cols, index=transactions.index).astype(int)
    transactions = pd.concat([transactions, encoded_cols_df], axis=1)
    transactions.drop(columns=['use_chip'], inplace=True)
    
    # One-hot encode 'errors' column
    errors_encoded = encoder.fit_transform(transactions[['errors']])
    encoded_errors_cols = encoder.get_feature_names_out(['errors'])
    encoded_errors_cols_df = pd.DataFrame(errors_encoded, columns=encoded_errors_cols, index=transactions.index).astype(int)
    transactions = pd.concat([transactions, encoded_errors_cols_df], axis=1)
    transactions.drop(columns=['errors'], inplace=True)

    # Clean column names
    transactions.columns = (
        transactions.columns
        .str.replace('use_chip_', '', regex=False)
        .str.replace('errors_', '', regex=False)
        .str.lower()
        .str.replace(' ', '_', regex=False)
    )

    return transactions

In [110]:
transactions = clean_transactions(df)

transactions.head()

,id,date,client_id,card_id,amount,mcc,chip_transaction,online_transaction,swipe_transaction,bad_cvv,...,bad_pin,"bad_pin,insufficient_balance","bad_pin,technical_glitch",bad_zipcode,"bad_zipcode,insufficient_balance","bad_zipcode,technical_glitch",insufficient_balance,"insufficient_balance,technical_glitch",none,technical_glitch
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,5499,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
1,7475328,2010-01-01 00:02:00,561,4575,14.57,5311,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
2,7475329,2010-01-01 00:02:00,1129,102,80.00,4829,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
3,7475331,2010-01-01 00:05:00,430,2860,200.00,4829,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
4,7475332,2010-01-01 00:06:00,848,3915,46.41,5813,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
